# Lab 1: Observability & Evaluation

In this lab, we tackle the **Black Box Problem**: understanding what happens inside an agent's reasoning loop and then objectively scoring its performance.

1. **Part 1: Tracing (The "See" Phase)**: Build a custom agent tracer, diagnose infinite loops, and use an infrastructure-free observer.
2. **Part 2: Evaluation (The "Score" Phase)**: Implement automated metrics to verify task completion and tool accuracy.

## Setup
Install dependencies and load environment variables.

In [1]:
# Run this cell first

import os
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import Optional
import litellm

# 1. Mute DeepEval warnings for a cleaner output
logging.getLogger("deepeval").setLevel(logging.ERROR)
os.environ["DEEPEVAL_TELEMETRY"] = "0"
os.environ["CONFIDENT_AI_API_KEY"] = "dummy"

load_dotenv()



---

# Part 1: Tracing (The "See" Phase)

Tracing allows us to see inside the "Black Box" of an agentic loop. Before using a professional tool, let's build a minimal `AgentTracer` to understand what we are actually capturing.

In [2]:
@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str] = None
    tool_calls: list = field(default_factory=list)
    cost_usd: float = 0.0

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    steps: list = field(default_factory=list)
    status: str = 'running'
    total_cost_usd: float = 0.0

class AgentTracer:
    def __init__(self):
        self.traces = {}
    def start_trace(self, trace_id, agent_name):
        self.traces[trace_id] = Trace(trace_id=trace_id, agent_name=agent_name)
        return trace_id
    def log_step(self, trace_id, step):
        if trace_id in self.traces:
            self.traces[trace_id].steps.append(step)
            self.traces[trace_id].total_cost_usd += step.cost_usd
    def print_trace(self, trace_id):
        t = self.traces.get(trace_id)
        if not t: return
        print(f'\n=== TRACE: {t.trace_id} === Agent: {t.agent_name}')
        for step in t.steps:
            print(f'  Step {step.step_number}: cost=${step.cost_usd:.4f}')
            for tc in step.tool_calls:
                print(f'    -> {tc.tool_name}({tc.tool_input}) [{tc.duration_ms:.0f}ms]')

tracer = AgentTracer()
print('AgentTracer ready.')

AgentTracer ready.


### Diagnosis: The Infinite Loop

Agents sometimes get stuck in "loops" where they call the same tool with the same input repeatedly. Let's simulate a broken agent and see how tracing helps diagnose it.

In [28]:
def search(query):
    time.sleep(0.5)
    return 'Error: Database timeout. Please try again.'

def run_agent(query, tracer, max_steps=3):
def run_agent(query, tracer, max_steps=3):
    trace_id = str(uuid.uuid4())[:8]
    tracer.start_trace(trace_id, 'broken_react_agent')
    messages = [{'role': 'user', 'content': query}]
    tools = [{'type': 'function', 'function': {'name': 'search', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}}}}]
    
    for step_num in range(1, max_steps + 1):
        response = litellm.completion(model='ollama/llama3.2', messages=messages, tools=tools, temperature=0)
        msg = response.choices[0].message
        messages.append(msg.model_dump())
        cost = 0.0 # simplified for demo
        step_record = AgentStep(step_number=step_num, reasoning=msg.content, cost_usd=cost)
        if not msg.tool_calls:
            tracer.log_step(trace_id, step_record)
            return msg.content, trace_id
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            t0 = time.time()
            result = search(args['query'])
            step_record.tool_calls.append(ToolCallRecord('search', args, result, (time.time()-t0)*1000))
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        tracer.log_step(trace_id, step_record)
    tracer.end_trace(trace_id, 'max_steps_reached')

    return 'Reached max steps.', trace_id

answer, tid = run_agent('What is the capital of France?', tracer)
tracer.print_trace(tid)


=== TRACE: fe2ab321 === Agent: broken_react_agent | Status: max_steps_reached
  Step 1: cost=$0.0000
    -> search({'query': 'capital of France'}) s]
  Step 2: cost=$0.0000
    -> search({'query': 'capital of France'}) s]
  Step 3: cost=$0.0000
    -> search({'query': 'capital of France'}) s]
Total Cost: $0.0000



### The Fix: Loop Detection
The trace shows the agent repeating itself. We can implement a `LoopDetector` to prevent this.

In [12]:
class LoopDetector:
    def __init__(self, threshold=2):
        self.history = []
        self.threshold = threshold
    def check(self, name, args):
        call = (name, args)
        count = self.history.count(call)
        self.history.append(call)
        return count >= self.threshold

detector = LoopDetector()
print('Is loop?', detector.check('search', '{"query": "France"}'))  # False
print('Is loop?', detector.check('search', '{"query": "France"}'))  # False
print('Is loop?', detector.check('search', '{"query": "France"}'))  # True after threshold

Is loop? False
Is loop? False
Is loop? False


### Professional Patterns: Simple Observer

Manual tracing is great for learning, but professional decorators handle timing and hierarchy automatically. We'll use a `SimpleObserver` that mimics the Langfuse API but runs locally without any infrastructure.

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric, ToolCorrectnessMetric
from deepeval.models import LiteLLMModel


@observe()
def think():
    time.sleep(0.8)  # Simulate LLM reasoning
    call_tool("Search")

@observe()
def agent_run():
    think()
    think() # Simulate a second turn

print('Starting evaluation...')
results = evaluate(test_cases=test_cases, metrics=[task_metric, tool_metric])
print(f'Done. {sum(1 for r in results if r.success)}/{len(results)} passed.')

# Part 2: Evaluation (The "Score" Phase)

Once we can see our traces, we need to **score** them. In this phase, we'll use **DeepEval** to run LLM-as-a-Judge metrics, verifying if our agent finished the task and used the correct tools.

In [27]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric, ToolCorrectnessMetric
from deepeval.models import LiteLLMModel

# Configure the judge model
judge_model = LiteLLMModel(model=MODEL_ID, temperature=0)

test_cases = [
    LLMTestCase(
        input='What is the capital of France?',
        actual_output='Paris is the capital of France.',
        expected_output='Paris',
        tools_called=[ToolCall(name='search', input_parameters={'query': 'capital of France'})],
        expected_tools=[ToolCall(name='search', input_parameters={'query': 'capital of France'})]
    )
]

    @observe(type='llm')
    def call_llm(self, messages, tools):
        return litellm.completion(model="openrouter/nvidia/nemotron-3-super-120b-a12b:free", messages=messages, tools=tools, temperature=0,tool_choice="auto")

    @observe(type='agent')
    def run(self, query, max_turns=2):
        print(f'[Agent] Running: {query}')
        messages = [{'role': 'user', 'content': query}]
        tools = [{
    'type': 'function', 
    'function': {
        'name': 'execute_search', 
        'description': 'ONLY use this for factual questions about world capitals or history', 
        'parameters': {
            'type': 'object', 
            'properties': {
                'query': {'type': 'string', 'description': 'The specific search term'}
            },
            'required': ['query']
        }
    }
}]
        for _ in range(max_turns):
            response = self.call_llm(messages, tools)
            msg = response.choices[0].message
            messages.append(msg.model_dump())
            
            if not msg.tool_calls:
                return msg.content
            for tc in msg.tool_calls:
                if self.detector.check(tc.function.name, tc.function.arguments): #stop the loop the tool has been called
                    return 'Agent stopped: loop detected.'
                args = json.loads(tc.function.arguments)
                result = self.execute_search(args['query'])
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

        return 'Max turns reached.'

agent = ObservableReactAgent()
print('Answer:', agent.run('What is the capital of France?'))

[Agent] Running: What is the capital of France?

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

  [Tool] Searching: capital of France

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

Answer: The capital of France is Paris.


### Summary: Forensic AI
1. **Trace (See)**: Use manual or professional observers to visualize the execution sequence.
2. **Diagnose**: Identify loops and failures early.
3. **Score (Evaluate)**: Use automated metrics like DeepEval to ensure quality at scale.